# Data processing – Predikce výnosu plodin v Indii

Tento notebook připravuje finální data pro HW2: crop data, weather join, leakage audit, chronologický split a uložení train/validation/test sad.

## 1. Task framing

Cílem projektu je predikovat `yield` zemědělských plodin v Indii na úrovni:

`State_Name + District_Name + Crop_Year + Season + Crop`.

Hlavní target je `yield`. Sloupec `Production` není používán jako modelová feature, protože je přímo navázaný na výnos a pěstovanou plochu.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path('data')
RAW = DATA / 'raw'
JOIN_KEYS = ['State_Name', 'District_Name', 'Crop_Year', 'Season', 'Crop']
WEATHER_COLS = [
    'rain_sum_mm', 'rainy_days_ge1mm', 'dry_days_lt1mm',
    'longest_dry_spell_days', 'longest_wet_spell_days',
    'heavy_rain_days_ge20mm', 'temp_mean_c', 'temp_max_mean_c',
    'temp_min_mean_c', 'hot_days_tmax_ge35c', 'humidity_mean_pct',
    'solar_mean', 'planting_dry_flag', 'harvest_too_wet_flag'
]

def clean_text_columns(df):
    df = df.copy()
    for col in ['State_Name', 'District_Name', 'Season', 'Crop']:
        if col in df.columns:
            df[col] = df[col].astype('string').str.strip().str.replace(r'\s+', ' ', regex=True)
    return df


## 2. Načtení dat

In [2]:
crop = pd.read_csv(RAW / 'Indian_crop_production_yield_dataset.csv', low_memory=False)
weather = pd.read_csv(DATA / 'weather_data_final.csv', low_memory=False)

crop = clean_text_columns(crop)
weather = clean_text_columns(weather)

print('crop:', crop.shape)
print('weather:', weather.shape)
display(crop.head())
display(weather.head())


crop: (575879, 8)
weather: (134263, 19)


,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production,yield
0,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Arecanut,1254.0,2000.0,1.594896
1,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Other Kharif pulses,2.0,1.0,0.500000
2,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Rice,102.0,321.0,3.147059
3,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Banana,176.0,641.0,3.642045
4,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Cashewnut,720.0,165.0,0.229167


,State_Name,District_Name,Crop_Year,Season,Crop,rain_sum_mm,rainy_days_ge1mm,dry_days_lt1mm,longest_dry_spell_days,longest_wet_spell_days,heavy_rain_days_ge20mm,temp_mean_c,temp_max_mean_c,temp_min_mean_c,hot_days_tmax_ge35c,humidity_mean_pct,solar_mean,planting_dry_flag,harvest_too_wet_flag
0,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Arecanut,978.01,133,20,11,52,9,27.62,27.86,27.31,0,83.30,17.02,0,1
1,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Other Kharif pulses,978.01,133,20,11,52,9,27.62,27.86,27.31,0,83.30,17.02,0,1
2,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Rice,978.01,133,20,11,52,9,27.62,27.86,27.31,0,83.30,17.02,0,1
3,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Banana,2106.62,279,87,11,52,18,27.46,27.74,27.12,0,82.07,17.25,1,1
4,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Cashewnut,2106.62,279,87,11,52,18,27.46,27.74,27.12,0,82.07,17.25,1,1


## 3. Popis dat

In [3]:
summary = pd.DataFrame({
    'missing_count': crop.isna().sum(),
    'missing_ratio': crop.isna().mean(),
    'n_unique': crop.nunique(dropna=True),
})
display(summary)
print('Crop years:', crop['Crop_Year'].min(), crop['Crop_Year'].max())
print('States:', crop['State_Name'].nunique())
print('District combinations:', crop[['State_Name','District_Name']].drop_duplicates().shape[0])
print('Crops:', crop['Crop'].nunique())
print('Duplicate rows:', crop.duplicated().sum())
display(crop['yield'].describe())


,missing_count,missing_ratio,n_unique
State_Name,0,0.0,36
District_Name,0,0.0,730
Crop_Year,0,0.0,24
Season,0,0.0,6
Crop,0,0.0,127
Area,0,0.0,48379
Production,0,0.0,113261
yield,0,0.0,337362


Crop years: 1997 2020
States: 36
District combinations: 796
Crops: 127


Duplicate rows: 31


count    5.758790e+05
mean     4.779376e+03
std      7.109329e+04
min      0.000000e+00
25%      1.346025e+00
50%      4.484848e+01
75%      1.314807e+02
max      4.395833e+06
Name: yield, dtype: float64

## 4. Weather join

Weather data jsou agregována na stejnou jednotku pozorování jako crop dataset. Denní weather data se nepřipojují přímo, aby nevznikla změna jednotky pozorování.

Připojení probíhá přes:

`State_Name, District_Name, Crop_Year, Season, Crop`.

In [4]:
weather = weather.drop_duplicates(subset=JOIN_KEYS)
joined = crop.merge(weather, on=JOIN_KEYS, how='left', validate='many_to_one')
joined.to_csv(DATA / 'crop_weather_joined.csv', index=False)

print('joined:', joined.shape)
print('rows preserved:', joined.shape[0] == crop.shape[0])
print('missing weather ratio:', joined['rain_sum_mm'].isna().mean())


joined: (575879, 22)
rows preserved: True
missing weather ratio: 0.6235615467832653


## 5. Leakage audit

- Target je `yield`.
- `Production` se nepoužívá jako feature, protože by vedl k target leakage.
- Weather features jsou agregované před joinem.
- Split je chronologický podle `Crop_Year`, ne náhodný.
- Preprocessing modelů v benchmarku je fitovaný pouze na train sadě.
- Test set se pouze vytvoří a uloží; nepoužívá se pro výběr modelu.

## 6. Finální weather-complete dataset a lag feature

In [5]:
df = joined.dropna(subset=['rain_sum_mm']).copy()
df = df[df['Crop_Year'].between(1997, 2014)].copy()
df = df.dropna(subset=['yield']).copy()

for col in ['Crop_Year', 'Area', 'Production', 'yield'] + WEATHER_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df['Crop_Year'] = df['Crop_Year'].astype(int)
df = df.sort_values(['State_Name', 'District_Name', 'Season', 'Crop', 'Crop_Year'])
df['lag_yield'] = df.groupby(['State_Name', 'District_Name', 'Season', 'Crop'])['yield'].shift(1)

df.to_csv(DATA / 'final_dataset_weather_complete.csv', index=False)
print('final weather-complete:', df.shape)
print('years:', df['Crop_Year'].min(), df['Crop_Year'].max())


final weather-complete: (186133, 23)
years: 1997 2014


## 7. Chronologický split

Použitý split:

- train: 1997–2010
- validation: 2011–2012
- test: 2013–2014

In [6]:
train = df[df['Crop_Year'].between(1997, 2010)].copy()
validation = df[df['Crop_Year'].between(2011, 2012)].copy()
test = df[df['Crop_Year'].between(2013, 2014)].copy()

train.to_csv(DATA / 'train.csv', index=False)
validation.to_csv(DATA / 'validation.csv', index=False)
test.to_csv(DATA / 'test.csv', index=False)

print('train:', train.shape, train['Crop_Year'].min(), train['Crop_Year'].max())
print('validation:', validation.shape, validation['Crop_Year'].min(), validation['Crop_Year'].max())
print('test:', test.shape, test['Crop_Year'].min(), test['Crop_Year'].max())


train: (143177, 23) 1997 2010
validation: (21349, 23) 2011 2012
test: (21607, 23) 2013 2014


## 8. Shrnutí

Výsledkem je finální weather-complete dataset a tři uložené datasety `train.csv`, `validation.csv` a `test.csv`. Test set zůstává stranou pro pozdější finální vyhodnocení.